In [ ]:
import os

data_dir = os.path.abspath("data/public")
project_dir = os.path.abspath("data/project")
chrombpnet_dir = os.path.join(project_dir, "chrombpnet")

In [ ]:
import seaborn as sns
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import torch
import io
from scipy.stats import mannwhitneyu

In [ ]:
class CPU_Unpickler(pickle.Unpickler):
    def find_class(self, module, name):
        # This is what pickle uses to rebuild torch storages
        if module == "torch.storage" and name == "_load_from_bytes":
            return lambda b: torch.load(io.BytesIO(b), map_location="cpu")
        return super().find_class(module, name)

path = f"{chrombpnet_dir}/data/motif_activity.pkl"
with open(path, "rb") as f:
    data = CPU_Unpickler(f).load()

In [ ]:
# Load motifs
with open(f"{chrombpnet_dir}/data/all_peaks.motifs", "rb") as f:
    hits = pickle.load(f)

In [ ]:
# Load all peaks
peaks = pd.read_csv(f"{chrombpnet_dir}/data/peaks_no_blacklist_summits.bed", sep="\t", header=None)

# Filter motifs
keep = []
for midx in range(len(hits)):
    nhits = len(set(hits[midx]['sequence_name'].tolist())) / len(peaks)
    if nhits >= 0.0001:
        if nhits <= 0.3:
            keep.append(midx)

hits_filtered = [hits[i] for i in keep]

In [ ]:
# Setup
celltypes = ["acinar", "activated_stellate", "alpha", "beta", "cycling", "delta","ductal", "endothelial", "epsilon", "lymphoid", "mast", "myeloid", "pp", "schwann", "quiescent_stellate"]
folds = [0,1,2,3,4]

In [ ]:
for midx in range(len(hits_filtered)):
    ma = []
    for celltype in celltypes:
        if len(data[celltype][midx][0]['after']) > 0:
            ma.append((data[celltype][midx][0]['before'] - data[celltype][midx][0]['after'].cpu().numpy()).mean())
        else:
            ma.append(np.nan)

    # Structure the data
    tmp_df = pd.DataFrame(np.array(ma), index = celltypes)
    tmp_df = tmp_df.rename(columns = {0: hits_filtered[midx]['motif_name'][0].split(" ")[0]}).T

    # Concat
    if midx == 0:
        ma_df = tmp_df
    else:
        ma_df = pd.concat([ma_df, tmp_df])

In [ ]:
ma_df = ma_df.dropna()

In [ ]:
ma_filtered = ma_df.copy()
ma_filtered = ma_filtered[~ma_filtered.index.str.contains("ZNF")]
ma_filtered = ma_filtered[~ma_filtered.index.str.contains("C2H2")]
ma_filtered = ma_filtered[ma_filtered.std(axis = 1) / ma_filtered.abs().mean(axis = 1) >= .75]
ma_filtered

In [ ]:
Z = ((ma_filtered.T - ma_filtered.T.mean()) / ma_filtered.T.std(ddof=0)).T

In [ ]:
sns.clustermap(Z)